In [1]:
import os
import json
import numpy as np
import onnx
import onnxruntime as ort
import tensorrt as trt
import pycuda.driver as cuda
import pycuda.autoinit

from PIL import Image
from tokenizers import Tokenizer


# =========================================================
# TOKENIZER
# =========================================================

class FastTokenizer:

    def __init__(self, tokenizer_blob):

        meta = tokenizer_blob.pop("meta")

        self.tokenizer = Tokenizer.from_str(
            json.dumps(tokenizer_blob)
        )

        self.special_ids = set(
            meta["special_ids"]
        )

        self.eos_token_id = (
            meta["eos_token_id"]
        )

    def decode(self, ids):

        ids = np.asarray(ids)

        if ids.ndim > 1:
            ids = ids.reshape(-1)

        eos_pos = np.where(
            ids == self.eos_token_id
        )[0]

        if eos_pos.size > 0:
            ids = ids[:eos_pos[0]]

        if self.special_ids:

            mask = ~np.isin(
                ids,
                list(self.special_ids)
            )

            ids = ids[mask]

        return self.tokenizer.decode(
            ids.tolist()
        )


# =========================================================
# FLORENCE2 HYBRID
# =========================================================

class Florence2Hybrid:

    def __init__(
        self,
        encoder_path,
        decoder_path,
        device="cuda",
        max_batch_size=8,
        max_new_tokens=15,
    ):

        self.encoder_path = encoder_path
        self.decoder_path = decoder_path

        self.engine_path = (
            encoder_path.replace(
                ".onnx",
                ".trt"
            )
        )

        self.max_batch_size = (
            max_batch_size
        )

        self.max_new_tokens = (
            max_new_tokens
        )

        # -------------------------------------------------
        # TOKENS
        # -------------------------------------------------

        self.bos_token_id = 0
        self.eos_token_id = 2

        # -------------------------------------------------
        # NORMALIZATION
        # -------------------------------------------------

        self.mean = np.array(
            [0.485, 0.456, 0.406],
            dtype=np.float32
        )

        self.std = np.array(
            [0.229, 0.224, 0.225],
            dtype=np.float32
        )

        self.rescale = 1.0 / 255.0

        self.image_size = 224

        # -------------------------------------------------
        # TOKENIZER
        # -------------------------------------------------

        for prop in onnx.load(
            decoder_path
        ).metadata_props:

            if prop.key == "tokenizer_json":

                self.tokenizer = (
                    FastTokenizer(
                        json.loads(prop.value)
                    )
                )

                break

        # -------------------------------------------------
        # BUILD ENGINE
        # -------------------------------------------------

        if not os.path.exists(
            self.engine_path
        ):

            self._build_engine()

        # -------------------------------------------------
        # TRT
        # -------------------------------------------------

        self.logger = trt.Logger(
            trt.Logger.ERROR
        )

        runtime = trt.Runtime(
            self.logger
        )

        with open(
            self.engine_path,
            "rb"
        ) as f:

            self.engine = (
                runtime.deserialize_cuda_engine(
                    f.read()
                )
            )

        self.context = (
            self.engine.create_execution_context()
        )

        self.stream = cuda.Stream()

        # -------------------------------------------------
        # ONNX DECODER
        # -------------------------------------------------

        providers = (
            [("CUDAExecutionProvider", {}), "CPUExecutionProvider"]
            if device == "cuda"
            else ["CPUExecutionProvider"]
        )

        sess_opts = ort.SessionOptions()
        sess_opts.enable_mem_pattern = False

        self.decoder_sess = ort.InferenceSession(
            decoder_path,
            sess_options=sess_opts,
            providers=providers,
        )

    # =====================================================
    # BUILD TRT ENGINE
    # =====================================================

    def _build_engine(self):

        print(
            "Building TensorRT encoder engine..."
        )

        logger = trt.Logger(
            trt.Logger.WARNING
        )

        builder = trt.Builder(
            logger
        )

        network = builder.create_network(
            1 << int(
                trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH
            )
        )

        parser = trt.OnnxParser(
            network,
            logger
        )

        with open(
            self.encoder_path,
            "rb"
        ) as f:

            if not parser.parse(
                f.read()
            ):

                errors = []

                for i in range(
                    parser.num_errors
                ):

                    errors.append(
                        str(
                            parser.get_error(i)
                        )
                    )

                raise RuntimeError(
                    "\n".join(errors)
                )

        config = (
            builder.create_builder_config()
        )

        config.set_memory_pool_limit(
            trt.MemoryPoolType.WORKSPACE,
            4 << 30
        )

        # -------------------------------------------------
        # FP16
        # -------------------------------------------------

        if builder.platform_has_fast_fp16:

            config.set_flag(
                trt.BuilderFlag.FP16
            )

        # -------------------------------------------------
        # FIXED SHAPES
        # -------------------------------------------------

        profile = (
            builder.create_optimization_profile()
        )

        profile.set_shape(
            "input_ids",
            (1, 10),
            (
                self.max_batch_size,
                10
            ),
            (
                self.max_batch_size,
                10
            ),
        )

        profile.set_shape(
            "pixel_values",
            (1, 3, 224, 224),
            (
                self.max_batch_size,
                3,
                224,
                224
            ),
            (
                self.max_batch_size,
                3,
                224,
                224
            ),
        )

        config.add_optimization_profile(
            profile
        )

        engine_bytes = (
            builder.build_serialized_network(
                network,
                config
            )
        )

        if engine_bytes is None:

            raise RuntimeError(
                "TRT build failed"
            )

        with open(
            self.engine_path,
            "wb"
        ) as f:

            f.write(engine_bytes)

        print(
            "Saved:",
            self.engine_path
        )

    # =====================================================
    # PREPROCESS
    # =====================================================

    def _preprocess_image(
        self,
        image,
    ):

        if image.mode != "RGB":

            image = image.convert(
                "RGB"
            )

        image = image.resize(
            (
                self.image_size,
                self.image_size,
            ),
            Image.Resampling.BICUBIC,
        )

        img = np.asarray(
            image,
            dtype=np.float32
        )

        img *= self.rescale
        img -= self.mean
        img /= self.std

        img = img.transpose(
            2,
            0,
            1
        )

        return np.ascontiguousarray(
            img[None]
        )

    # =====================================================
    # TRT ENCODER
    # =====================================================

    def _encoder_forward(
        self,
        input_ids,
        pixel_values,
    ):

        # -------------------------------------------------
        # FORCE TYPES
        # -------------------------------------------------

        input_ids = np.ascontiguousarray(
            input_ids.astype(np.int64)
        )

        pixel_values = np.ascontiguousarray(
            pixel_values.astype(np.float16)
        )

        # -------------------------------------------------
        # SET SHAPES
        # -------------------------------------------------

        self.context.set_input_shape(
            "input_ids",
            input_ids.shape
        )

        self.context.set_input_shape(
            "pixel_values",
            pixel_values.shape
        )

        # -------------------------------------------------
        # OUTPUT SHAPE
        # -------------------------------------------------

        output_shape = tuple(
            self.context.get_tensor_shape(
                "encoder_hidden_states"
            )
        )

        # -------------------------------------------------
        # HOST OUTPUT
        # -------------------------------------------------

        host_output = np.empty(
            output_shape,
            dtype=np.float16
        )

        # -------------------------------------------------
        # GPU BUFFERS
        # -------------------------------------------------

        d_input_ids = cuda.mem_alloc(
            input_ids.nbytes
        )

        d_pixel_values = cuda.mem_alloc(
            pixel_values.nbytes
        )

        d_output = cuda.mem_alloc(
            host_output.nbytes
        )

        # -------------------------------------------------
        # COPY INPUTS
        # -------------------------------------------------

        cuda.memcpy_htod_async(
            d_input_ids,
            input_ids,
            self.stream
        )

        cuda.memcpy_htod_async(
            d_pixel_values,
            pixel_values,
            self.stream
        )

        # -------------------------------------------------
        # BIND
        # -------------------------------------------------

        self.context.set_tensor_address(
            "input_ids",
            int(d_input_ids)
        )

        self.context.set_tensor_address(
            "pixel_values",
            int(d_pixel_values)
        )

        self.context.set_tensor_address(
            "encoder_hidden_states",
            int(d_output)
        )

        # -------------------------------------------------
        # EXECUTE
        # -------------------------------------------------

        success = (
            self.context.execute_async_v3(
                self.stream.handle
            )
        )

        if not success:

            raise RuntimeError(
                "TensorRT execution failed"
            )

        # -------------------------------------------------
        # COPY OUTPUT
        # -------------------------------------------------

        cuda.memcpy_dtoh_async(
            host_output,
            d_output,
            self.stream
        )

        self.stream.synchronize()

        return host_output

    # =====================================================
    # MAIN INFERENCE
    # =====================================================

    def infer_batch(
        self,
        images,
    ):

        batch_size = len(images)

        pixel_values = np.concatenate(
            [
                self._preprocess_image(img)
                for img in images
            ],
            axis=0
        ).astype(np.float16)

        # EXACT SAME PROMPT

        prompt = [
            0,
            2264,
            16,
            5,
            2788,
            11,
            5,
            2274,
            116,
            2
        ]

        input_ids = np.tile(
            np.array(
                prompt,
                dtype=np.int64
            )[None],
            (batch_size, 1)
        )

        # -------------------------------------------------
        # TRT ENCODER
        # -------------------------------------------------

        encoder_hidden = (
            self._encoder_forward(
                input_ids=input_ids,
                pixel_values=pixel_values,
            )
        )

        # -------------------------------------------------
        # EXACT SAME DECODER LOGIC
        # -------------------------------------------------

        generated_ids = np.full(
            (
                batch_size,
                self.max_new_tokens + 1
            ),
            fill_value=self.eos_token_id,
            dtype=np.int64
        )

        generated_ids[:, 0] = (
            self.bos_token_id
        )

        cur_len = 1

        finished = np.zeros(
            batch_size,
            dtype=bool
        )

        for _ in range(
            self.max_new_tokens
        ):

            logits = self.decoder_sess.run(
                ["logits"],
                {
                    "decoder_input_ids":
                        generated_ids[:, :cur_len],

                    "encoder_hidden_states":
                        encoder_hidden,
                },
            )[0]

            next_tokens = np.argmax(
                logits[:, -1, :],
                axis=-1
            )

            generated_ids[:, cur_len] = (
                next_tokens
            )

            cur_len += 1

            finished |= (
                next_tokens
                == self.eos_token_id
            )

            if finished.all():
                break

        outputs = []

        for i in range(batch_size):

            outputs.append(
                self.tokenizer.decode(
                    generated_ids[i]
                )
            )

        return outputs

In [ ]:
# for some reason initializing this speeds up the token decoder
from transformers import AutoProcessor
processor = AutoProcessor.from_pretrained("./weights/model_best_st_224x224/", trust_remote_code=True) 

In [ ]:
from PIL import Image
from IPython.display import display
import glob

# ---------------------------------------------------------
# INIT MODEL
# ---------------------------------------------------------

model = Florence2Hybrid(
    encoder_path="florence2_encoder_fp16.onnx",
    decoder_path="florence2_decoder_fp16.onnx",
    device="cuda",
    max_batch_size=8,
    max_new_tokens=16,
)

# ---------------------------------------------------------
# LOAD IMAGES
# ---------------------------------------------------------

images_path = glob.glob(
    "./dataset/lp_crops_st_balanced/val/*.jpg"
)

# ---------------------------------------------------------
# SINGLE IMAGE TEST
# ---------------------------------------------------------

for image_path in images_path[:5]:

    with open(
        image_path.replace(".jpg", ".txt")
    ) as f:

        gt = f.read().strip()

    image = Image.open(
        image_path
    ).convert("RGB")

    pred = model.infer_batch(
        [image]
    )[0]

    display(image)

    print("gt :", gt)
    print("pr :", pred)
    print()

    break

In [ ]:
import os
import glob

from tqdm import tqdm
from PIL import Image


# ---------------------------------------------------------
# CONFIG
# ---------------------------------------------------------

BATCH_SIZE = 8

model = Florence2Hybrid(
    encoder_path="florence2_encoder_fp16.onnx",
    decoder_path="florence2_decoder_fp16.onnx",
    device="cuda",
    max_batch_size=BATCH_SIZE,
    max_new_tokens=16,
)


# ---------------------------------------------------------
# LEVENSHTEIN
# ---------------------------------------------------------

def levenshtein_distance(s1, s2):

    if len(s1) < len(s2):

        return levenshtein_distance(
            s2,
            s1
        )

    if len(s2) == 0:
        return len(s1)

    prev_row = list(
        range(len(s2) + 1)
    )

    for i, c1 in enumerate(s1):

        curr_row = [i + 1]

        for j, c2 in enumerate(s2):

            insertions = (
                prev_row[j + 1] + 1
            )

            deletions = (
                curr_row[j] + 1
            )

            substitutions = (
                prev_row[j]
                + (c1 != c2)
            )

            curr_row.append(
                min(
                    insertions,
                    deletions,
                    substitutions
                )
            )

        prev_row = curr_row

    return prev_row[-1]


# ---------------------------------------------------------
# METRICS
# ---------------------------------------------------------

total = 0

d0 = 0
d1 = 0
d2 = 0

# ---------------------------------------------------------
# DATASET
# ---------------------------------------------------------

images_path = glob.glob(
    "./dataset/lp_crops_st_balanced/val/*.jpg"
)

batch_images = []
batch_gts = []

# ---------------------------------------------------------
# LOOP
# ---------------------------------------------------------

for image_path in tqdm(images_path):

    gt_path = image_path.replace(
        ".jpg",
        ".txt"
    )

    if not os.path.exists(gt_path):
        continue

    with open(gt_path) as f:

        gt = f.read().strip()

    if not gt:
        continue

    if ("\"" in gt) or (" " in gt):
        continue

    # -----------------------------------------------------
    # LOAD IMAGE
    # -----------------------------------------------------

    image = Image.open(
        image_path
    ).convert("RGB")

    batch_images.append(image)
    batch_gts.append(gt)

    # -----------------------------------------------------
    # RUN BATCH
    # -----------------------------------------------------

    if len(batch_images) == BATCH_SIZE:

        preds = model.infer_batch(
            batch_images
        )

        for gt, pred in zip(
            batch_gts,
            preds
        ):

            pred = pred.replace(
                " ",
                ""
            ).strip()

            dist = levenshtein_distance(
                gt.lower(),
                pred.lower()
            )

            total += 1

            if dist == 0:
                d0 += 1

            if dist <= 1:
                d1 += 1

            if dist <= 2:
                d2 += 1

        batch_images.clear()
        batch_gts.clear()

# ---------------------------------------------------------
# LEFTOVER BATCH
# ---------------------------------------------------------

if batch_images:

    preds = model.infer_batch(
        batch_images
    )

    for gt, pred in zip(
        batch_gts,
        preds
    ):

        pred = pred.replace(
            " ",
            ""
        ).strip()

        dist = levenshtein_distance(
            gt.lower(),
            pred.lower()
        )

        total += 1

        if dist == 0:
            d0 += 1

        if dist <= 1:
            d1 += 1

        if dist <= 2:
            d2 += 1

# ---------------------------------------------------------
# RESULTS
# ---------------------------------------------------------

acc_exact = d0 / total
acc_1 = d1 / total
acc_2 = d2 / total

print()
print(f"Total samples              : {total}")
print(f"Exact accuracy (dist=0)    : {acc_exact:.4f}")
print(f"Accuracy (dist<=1)         : {acc_1:.4f}")
print(f"Accuracy (dist<=2)         : {acc_2:.4f}")
print()